# jupyterlab-unsafe-globals demo

[jupyterlab-unsafe-globals](https://github.com/jtpio/jupyterlab-unsafe-globals) restores the classic Notebook globals — `window.Jupyter` and its alias `window.IPython` — on top of the modern JupyterLab APIs, in JupyterLab and Jupyter Notebook 7+.

This notebook exercises the shims the way classic scripts did: from `%%javascript` cells. Two things to keep in mind:

- ⚠️ **This is unsafe by design.** The classic globals hand any JavaScript running on the page full control over the notebook, the kernel, and the application — that is why Jupyter Notebook 7+ and JupyterLab removed them on purpose, and what this extension opts back into. If a cell output shows a *Run* button instead of executing, that is the standard JupyterLab trust machinery asking for confirmation — click it to proceed.
- In the classic Notebook, `element` in a `%%javascript` cell was a jQuery object; in JupyterLab it is a plain DOM element, so this notebook uses `element.textContent` instead of `element.text(...)`.

## The globals are back

The extension puts `Jupyter` on the `window`, with `IPython` as a plain alias, like in classic.

In [ ]:
%%javascript
element.textContent =
  `window.Jupyter is back (version ${Jupyter.version}), ` +
  `and window.IPython is the same object: ${window.IPython === window.Jupyter}`;

## Drive the notebook

`Jupyter.notebook` points to the current notebook and exposes the classic cell API: insert, select, move, copy and paste, convert, execute…

In [ ]:
%%javascript
const nb = Jupyter.notebook;
const cell = nb.insert_cell_below('code');
cell.set_text('print("hello from a cell inserted with Jupyter.notebook")');
cell.execute();
element.textContent = `cell inserted and executed; the notebook now has ${nb.ncells()} cells`;

## Run code on the kernel, classic callbacks included

`Jupyter.notebook.kernel.execute` takes the exact classic callback shape, with the classic defaults (`silent: true`, `store_history: false`). This site runs the [Pyodide](https://pyodide.org) kernel, so the answer comes from a Python interpreter running in your browser — note the `emscripten` platform.

In [ ]:
%%javascript
element.textContent = 'the kernel says: ';
Jupyter.notebook.kernel.execute(
  'import sys; print(f"Python {sys.version.split()[0]} on {sys.platform}")',
  {
    shell: {
      reply: msg => console.log('execute_reply status:', msg.content.status)
    },
    iopub: {
      output: msg => {
        element.textContent += msg.content.text ?? '';
      }
    }
  }
);

## Add toolbar buttons

`Jupyter.toolbar.add_buttons_group` was the classic way for extensions to add buttons, with FontAwesome 4 icons. The shim adds the group to the toolbar of every open and future notebook. Run the next cell once, then look at the notebook toolbar: the clock button asks the kernel for the time and shows it with `Jupyter.dialog.modal`.

In [ ]:
%%javascript
Jupyter.toolbar.add_buttons_group([
  {
    label: 'Save',
    icon: 'fa-save',
    help: 'save the notebook, classic style',
    callback: () => Jupyter.notebook.save_notebook()
  },
  {
    icon: 'fa-clock-o',
    help: 'ask the kernel for the time',
    callback: () => {
      Jupyter.notebook.kernel.execute(
        'import datetime; print(datetime.datetime.now().isoformat())',
        {
          iopub: {
            output: msg => {
              const text = (msg.content.text ?? '').trim();
              if (text) {
                Jupyter.dialog.modal({
                  title: 'Kernel time',
                  body: text,
                  buttons: { OK: {} }
                });
              }
            }
          }
        }
      );
    }
  }
]);
element.textContent = 'button group added to the notebook toolbar (look up)';

## Listen to classic events

`Jupyter.events` emits over 30 classic events with the jQuery `on`/`one`/`off`/`trigger` conventions. Run the next cell, then run any code cell to see the counter move.

In [ ]:
%%javascript
let count = 0;
element.textContent = 'waiting for finished_execute.CodeCell events…';
Jupyter.events.on('finished_execute.CodeCell', (event, data) => {
  count += 1;
  element.textContent =
    `finished_execute.CodeCell fired ${count} time(s), ` +
    `last cell started with: ${data.cell.get_text().split('\n')[0]}`;
});

## Add keyboard shortcuts

Classic shortcut strings work, bound through the application command registry. Run the next cell, click a cell prompt to enter command mode, then press `Ctrl-Shift-H`.

In [ ]:
%%javascript
Jupyter.keyboard_manager.command_shortcuts.add_shortcut('ctrl-shift-h', () => {
  Jupyter.dialog.modal({
    title: 'Hello from a classic shortcut',
    body: 'This dialog was opened by a shortcut added with Jupyter.keyboard_manager.command_shortcuts.add_shortcut.',
    buttons: { Close: {} }
  });
});
element.textContent = 'press Ctrl-Shift-H in command mode';

## The kernel can drive the page

This is the pattern the warning in the README is about: code running on the kernel injects JavaScript into the page with `IPython.display.Javascript`. Classic extensions and snippets used it heavily; with this extension installed it works again.

In [ ]:
from IPython.display import Javascript, display

display(Javascript("""
const cell = Jupyter.notebook.insert_cell_below('markdown');
cell.set_text('**This cell was inserted and rendered by Python code running on the kernel.**');
cell.render();
"""))

## The migration escape hatch

`Jupyter.app` is the `JupyterFrontEnd` application — not part of classic, provided so scripts can migrate to the real [JupyterLab APIs](https://jupyterlab.readthedocs.io/en/latest/extension/extension_dev.html) one call at a time.

In [ ]:
%%javascript
element.textContent =
  `Jupyter.app is the JupyterFrontEnd application: ` +
  `${Jupyter.app.name} ${Jupyter.app.version}, ` +
  `with ${Jupyter.app.commands.listCommands().length} commands`;

## What else

See the [README](https://github.com/jtpio/jupyterlab-unsafe-globals#what-is-shimmed) for the full list of what is shimmed — the cell clipboard, checkpoints, `Jupyter.contents`, the comm manager, the CodeMirror 5 adapter — and for what is deliberately out of scope. And remember: install this only if you understand and accept the classic security posture it restores.